---
---
# **Tutorial 1A:** *ML basics → Neural Networks → the Gemini API*
---
---

### QUESTIONS FOR TODAY
> 1.   *Why are we integrating ML Models into System Architecture !!*
> 2.   *Why we cannot treat ML as a "Black Box" ??*

### OUR ROADMAP (Tracking the Parameter Scale)
|Stage |What We Are Building	|Scale (Parameter Count)|
|---|---|---|
|1. Regression |Straight line that learns	|2 |
|2. Classification |Spam detector |2 |
|3. Neural Network |MLP mapping patterns |~340 |
|4. Gemini API |Call a trillion-parameter model over HTTPS |~10¹² |

### TASK FOR TODAY
> **PREDICT → MEASURE THE ERROR → ADJUST THE PARAMETERS → REPEAT.**

---
# **Part 0: The Machine Learning Landscape**
---

MACHINE LEARNING ≈ *programming with examples instead of rules*. The machine learns !

Depending on what kind of examples you have, ML is generally categorized into three distinct approaches:

| Learning | Core Concept | Training Data Provided | Industry Examples |
|---|---|---|---|
| **Supervised** | *Learns from established examples to map inputs to known outputs.* | Inputs paired with correct target outputs (labels). | spam filter, price prediction |
| **Unsupervised** | *Discovers hidden structures, patterns, or groupings within messy data.* | Raw inputs only (No predefined labels). | customer segmentation, anomaly detection |
| **Reinforcement** | *Optimizes a sequence of actions through continuous trial and error.* | An environment governed by a reward and penalty mechanism. | game bots, robot control |

---

### ❓ Quick Audience Poll ------------------------------------
> Please type **1 (Supervised)**, **2 (Unsupervised)**, or **3 (Reinforcement)** in the chat for each scenario:

> **A.**  Predicting tomorrow's server load based on historical traffic data. <br>
**B.**  Grouping platform users into distinct "personas" without predefined categories. <br>
**C.**  A bot that learns chess by winning/losing millions of games. <br>
**D.**  Large Language Models like ChatGPT / Gemini / Claude

---

<details><summary><b>Reveal answers</b> (click)</summary>

**A** = supervised (past traffic comes with the "answer" as the load has actually happened). <br>
**B** = unsupervised. <br>
**C** = reinforcement. <br>
**D** is the trick one: the precise answer is **self-supervised**. LLMs are mostly **supervised** where the task is to *"predict the next word"*, and the internet conveniently contains its own answers (every sentence is a flashcard!). Then a layer of **reinforcement** learning (RLHF) teaches them to be helpful rather than just autocomplete. Two learnings in one.
</details>

### The imports we need for the ML!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)   # same "random" numbers for everyone for reproducing!
print("Ready to learn ML.")

---
# **Part 1: Linear Regression - *"Hello World" of machine learning***
---

*Scenario:* You've logged some data say, **marketing spend (x) vs. revenue (y)** and you suspect a linear relationship between them. We can represent this with the standard equation for a straight line:

$$y = m \cdot x + c$$

In this equation, we have two adjustable parameters: the slope ($m$) and the intercept ($c$). <br>
> *In the context of machine learning, "training" simply means mathematically finding the optimal values for these two parameters so that the line best fits your historical data.* <br>

### Let's generate some sample data to demonstrate this:

In [ ]:
def make_points(m=5, c=10, n=15, noise=6):
    '''Generate synthetic data along the baseline equation y = 5x + 10,
    adding some noise to simulate real-world variance.'''
    x = np.linspace(1, 10, n)
    y = m * x + c + np.random.normal(0, noise, n)   # baseline truth + noise
    return x, y

x, y = make_points()

#print("x [input]: ",x)
#print("y [output]: ",x)

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y, color="m", s=50)
plt.xlabel("x (Independent Variable: Marketing Spend)")
plt.ylabel("y (Dependent Variable: Revenue)")
plt.title("Historical Data: Identifying the Underlying Linear Trend")
plt.show()

### You *being the learning algorithm*: Manual Optimization

In the widget below, you will act as the learning algorithm. Move the sliders to adjust the slope ($m$) and intercept ($c$) to fit the prediction line to our historical data.

Your score-**the Loss**—is calculated in real-time using the MSE formula:

$$\text{MSE} = \frac{1}{n}\sum (y_{\text{actual}} - y_{\text{predicted}})^2$$

> **The Challenge:** Adjust the parameters until you reduce the **Loss (MSE) to below 40.**

In [ ]:
from ipywidgets import interact

@interact(m=(-2.0, 12.0, 0.5), c=(-20.0, 40.0, 2.0))
def manual_optimization(m=0.0, c=0.0):
    """
    Interactive widget simulating the optimization of parameters m and c.
    """
    # 1. Calculate predictions based on current slider values
    y_pred = m * x + c

    # 2. Calculate the Loss (Mean Squared Error)
    mse = np.mean((y - y_pred) ** 2)

    # 3. Visualize the data and the prediction line
    plt.figure(figsize=(7, 4))
    plt.scatter(x, y, color="m", s=50, label="Historical Data")
    plt.plot(x, y_pred, "g", linewidth=2, label=f"Prediction Line (y = {m}x + {c})")

    # Formatting the chart
    plt.ylim(-15, 80)
    plt.xlabel("x (Independent Variable: Marketing Spend)")
    plt.ylabel("y (Dependent Variable: Revenue)")
    plt.title(f"Current Loss (MSE): {mse:,.1f}   |   Target Loss: < 40", fontweight='bold')
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

### Gradient Descent: Automating the Optimization

Think about the strategy you just used: you adjusted a parameter, observed the change in the Loss, and if the score worsened, you moved in the opposite direction.

>**Congratulations**!! <br>
You just manually executed an algorithm known as **Gradient Descent**.

In modern software architecture, we *automate* this exact process. Here's how:

1.	**Predict**: *Generate outputs using the current parameters ($\hat{y} = mx + c$).*
2.	**Measure**: *Calculate the current Loss (MSE)*.
3.	**Adjust**: *Determine which mathematical direction reduces the error for each parameter, and take a small step in that direction.*
4.	**Repeat**: *Repeat this cycle until the loss is minimized.*
---

***The Gradient and The Learning Rate***

> The mathematical mechanism determining "*which way reduces the error*" is the **Gradient**.

> Imagine standing on a steep hillside in thick fog: you cannot see the valley floor, but you can feel the slope of the ground beneath your feet and step downward. The size of the step you take is controlled by a configuration called the **Learning Rate**:
> *   *If the learning rate is too large, you risk overshooting the valley floor entirely and destabilizing the model.*
> *   *If the learning rate is too small, the algorithm takes microscopic steps, wasting expensive compute time to reach the bottom.*

The two `d`-lines (derivatives) below are the only calculus in this entire course and even they're optional. All those scary formulas in ML papers exist for exactly one purpose: *which way reduces the error.*

### The Code: Building the Optimization Engine

In [ ]:
def gradient_descent_step(x, y, m, c, lr):
    """
    Executes a single iteration of Gradient Descent to optimize m and c.
    """
    # 1. PREDICT: Generate predictions using current parameters
    y_pred = m * x + c

    # 2. MEASURE: Calculate the current error (Loss)
    loss = np.mean((y - y_pred) ** 2)

    # CALCULATE GRADIENTS: The 'd-lines' (derivatives) of m and c showing which way reduces the error
    dm = -2 * np.mean(x * (y - y_pred))
    dc = -2 * np.mean(y - y_pred)

    # 3. ADJUST: Take a small step scaled by the learning rate (lr)
    m_new = m - lr * dm
    c_new = c - lr * dc

    return m_new, c_new, loss

### The Training Loop # 4. REPEAT

In [ ]:
# Initialize parameters at zero (starting with no knowledge)
m, c = 0.0, 0.0
losses = []            # An array to track our error over time

# The Training Loop: Iterate 200 times
for step in range(200):
    m, c, loss = gradient_descent_step(x, y, m, c, lr=0.01)
    losses.append(loss)

    # Log the system's progress periodically
    if step % 40 == 0 or step == 199:
        print(f"Step {step:3d} | Slope (m) = {m:5.2f} | Intercept (c) = {c:5.2f} | Loss = {loss:8.1f}")

In [ ]:
# ---------------------------------------------------------
# Visualization: The Results Dashboard
# ---------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Left Panel: How well does the final line fit our data?
ax[0].scatter(x, y, color="m", s=50, label="Historical Data")
ax[0].plot(x, m * x + c, "g", linewidth=2, label="Automated Prediction Line")
ax[0].set_title(f"Machine's Fit: y = {m:.2f}x + {c:.2f}  (Actual Baseline Truth: 5x+10)")
ax[0].legend()

# Right Panel: Tracking the error over time
ax[1].plot(losses, color="tab:red", linewidth=2)
ax[1].set_xlabel("Training Iteration (Step)")
ax[1].set_ylabel("Loss (MSE)")
ax[1].set_title("The Learning Curve (Loss Optimization)")
ax[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

Take a look at the machine's final Loss score and compare it to your manual attempt.

> We just optimized a model with exactly two parameters using a 4-step engine: <br>
***Predict, Measure, Adjust, Repeat***.

Keep this core loop in mind. The math doesn't fundamentally change; we are just scaling up the parameter count.


---
# **Part 2: Classification - *predicting a category***
---

Regression predicts a **number**. But most business problems require a **category**.

Instead of asking "*How much?*", we are asking binary questions: *Is this email spam or legitimate?* *Should this transaction be approved or flagged as fraud?*

> Here's the classification engine, in three steps:
> 1. **Calculate a Raw Score**: We compute our standard linear equation ($z = m x + c$) (higher score → more spam-like)
> 2. **Transform into Probability**: push the score through a **sigmoid**, an S-shaped math function that compresses any number into the strict range **0 to 1**. *"compress any score into a probability!"*
> 3. **Set a Decision Threshold**: We define a cutoff point. For example, if the probability is greater than 0.5 (50%), the system classifies the email as "Spam."

> This is **Logistic Regression** — which, in the greatest naming failure in ML history, is a *classification* algorithm.
(It's called that because it's exactly what we just described: standard regression with a probability transformation applied on top.)

### Setup
Let's build a spam detector from one feature (*using a single independent variable*):

*The frequency of flagged keywords in an email?* (e.g., "FREE", "WINNER", "URGENT")

In [ ]:
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# 1. Generate Synthetic Data
# ---------------------------------------------------------
rng = np.random.default_rng(7)

# Simulating the feature: Flagged keyword counts in 80 emails
legit_emails = rng.poisson(1.2, 40)          # Legitimate: Average ~1 flagged word
spam_emails  = rng.poisson(6.0, 40) + 1      # Spam: Higher frequency of flagged words

X = np.concatenate([legit_emails, spam_emails]).reshape(-1, 1) # Feature array (Inputs)

# Defining our target categories (0 = Legitimate, 1 = Spam)
y = np.array([0] * 40 + [1] * 40)

print("X",X)
#print("y",y)

In [ ]:
# ---------------------------------------------------------
# 2. Model Training
# ---------------------------------------------------------
# The .fit() method automatically runs the entire gradient descent loop internally
clf = LogisticRegression().fit(X, y)

In [ ]:
# ---------------------------------------------------------
# 3. Visualization: The Probability Curve
# ---------------------------------------------------------
# Generate a smooth line of x-values to draw the curve
xs = np.linspace(0, 12, 200).reshape(-1, 1)

plt.figure(figsize=(8, 4.5))

# Plot the historical email data (Adding minor vertical noise/jitter so dots don't overlap)
plt.scatter(X, y + rng.normal(0, 0.02, len(y)), c=y, cmap="coolwarm", alpha=0.7, s=40)

# Plot the Sigmoid S-Curve
plt.plot(xs, clf.predict_proba(xs)[:, 1], "g", linewidth=2, label="Probability Curve (Sigmoid)")

# Plot the Decision Boundary
plt.axhline(0.5, color="gray", ls="--", label="Decision Threshold (50%)")

# Formatting the Dashboard
plt.xlabel("Flagged Keywords Count (Independent Variable)")
plt.ylabel("Probability of Spam (Dependent Variable)")
plt.title("Logistic Regression: Probability Curve & Decision Boundary", fontweight="bold")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# ---------------------------------------------------------
# 4. Evaluation and Inference Testing
# ---------------------------------------------------------
print(f"Overall Model Accuracy: {clf.score(X, y):.0%}\n")

print("Live Inference Examples:")
for n_words in [1, 4, 7]:
    # Extracting the probability of the positive class (Spam)
    p = clf.predict_proba([[n_words]])[0, 1]
    print(f"Email with {n_words} flagged words  -->  Probability of Spam: {p:.0%}")

### ❓ Quick Audience Poll ------------------------------------
> Please type **1 (Regression)** or **2 (Classification)** in the chat for each scenario:

> **A.**  Predicting a house's sale price <br>
**B.**  Flagging a transaction as fraud <br>
**C.**  Estimating delivery ETA in minutes <br>
**D.**  Routing a support ticket to the right team

---

<details><summary><b>Reveal answers</b> (click)</summary>

**A** = regression. <br>
**B** = classification. <br>
**C** = regression. <br>
**D** is the fun one: classification (multi-class — same trick, one score per team, pick the highest).
</details>

Notice: the model doesn't say *"spam"*. It says *"99% spam"* — **models output probabilities; the threshold is a business decision.**

> Blocking a real email is worse than letting one spam through? Then don't threshold at 0.5, threshold at 0.9. That's an *architecture* choice, not a math one.

**Parameter counter: still 2.** It is the exact same linear engine we built in PART 1, just with a probability transformation applied on top.

*But there's a problem coming.....*

---
# **Part 3: Neural Networks - *overcoming limitations***
---

Our spam detector draws a **straight** decision boundary. However, real-world data is rarely that simple.

The **patterns** found in financial fraud, image recognition, and human language are highly complex, interdependent, and fundamentally **non-linear**.

> Let's observe what happens when a standard linear model encounters non-linear data (*the classic "two moons" dataset*), and then watch how a fundamental **Neural Network** easily adapts to map those complex, curved boundaries.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# 1. Generate Non-Linear Data
# ---------------------------------------------------------
# Using the "Two Moons" dataset to simulate complex, overlapping data boundaries
X2, y2 = make_moons(n_samples=300, noise=0.25, random_state=42)

In [ ]:
# ---------------------------------------------------------
# 2. Train the Models
# ---------------------------------------------------------
# Model A: Standard Linear Baseline
linear_model = LogisticRegression().fit(X2, y2)

# Model B: Neural Network (Multi-Layer Perceptron)
# Architecture: Two hidden layers with 16 neurons each
mlp_model = MLPClassifier(hidden_layer_sizes=(16, 16),
                          max_iter=3000,
                          random_state=42).fit(X2, y2)

In [ ]:
# ---------------------------------------------------------
# 3. Visualization Helper Function
# ---------------------------------------------------------
def plot_decision_boundary(model, X, y, title, ax):
    """Generates a contour plot visualizing the model's decision boundary."""
    h = 0.02
    xx, yy = np.meshgrid(np.arange(X[:, 0].min() - .5, X[:, 0].max() + .5, h),
                         np.arange(X[:, 1].min() - .5, X[:, 1].max() + .5, h))

    # Predict every point on the grid to create the background boundary regions
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title, fontweight="bold")

In [ ]:
# ---------------------------------------------------------
# 4. Render the Comparison Dashboard
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_decision_boundary(linear_model, X2, y2,
                       f"Linear Model Boundary | Accuracy: {linear_model.score(X2, y2):.0%}",
                       axes[0])

plot_decision_boundary(mlp_model, X2, y2,
                       f"Neural Network Boundary | Accuracy: {mlp_model.score(X2, y2):.0%}",
                       axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# 5. Tracking the Scale (Parameter Count)
# ---------------------------------------------------------
# Total parameters = (weights) + (biases/intercepts)
n_params = sum(w.size for w in mlp_model.coefs_) + sum(b.size for b in mlp_model.intercepts_)
print(f"Total trainable parameters in this Neural Network: {n_params}")

### What's inside the Neural Network Architecture?

An **MLP** (Multi-Layer Perceptron) might sound complex, but it is fundamentally built from the exact same mathematical concepts we have already covered:

* A **Node** (or **Neuron**) = essentially *a linear regression equation* ($w_1x_1 + w_2x_2 + \dots + b$) plus *a  non-linear transformation* (*activation function*).
* A **Layer** = a horizontal arrangement of *independent neurons* processing data working *in parallel*.
* A **Network** = a vertical stack of *interconnected layers*. Early layers identify raw, fundamental patterns, while deeper layers combine those findings into highly sophisticated representations.

---

> **Q: Why do we need that mathematical "bend" (the activation function)?**

If you stack multiple linear layers together without introducing non-linearity, the entire architecture mathematically collapses back into a single straight line. Those non-linear transformations are exactly what allow the model to bend its decision boundary to fit complex real-world data.

> **Q: How is it trained? (Backpropagation)**

The engine remains identical to our previous exercise:  *predict → measure → adjust → repeat.* The only architectural addition is **backpropagation** — an efficient computational technique that calculates the gradient for hundreds of parameters simultaneously rather than just two. The optimization landscape is larger, but the mechanics are identical.

> **Architect's note on Scale:** Moving forward, asking "*Which model size do I actually need to accomplish this task?*" will be one of your most critical engineering and design decisions.

**Parameter counter: 337.**

---
# **Part 4: *orchestrating foundation models via the* Gemini API**
---

> Q: **How do we jump from a 337-parameter neural network to a frontier model like Gemini?**

Here's the industry reality:
> **you will (almost) never train a frontier model from scratch. You will *call/orchestrate* one.**

### Testing the Microservice (Zero-Setup)

In [ ]:
# ---------------------------------------------------------
# Quick Test: Calling an LLM Microservice
# ---------------------------------------------------------
# Using Google Colab's built-in AI module for a quick, keyless test.

try:
    from google.colab import ai

    # Sending our 'request' to the model
    response = ai.generate_text("In one sentence, how would you define Agentic AI?")
    print(response)

except Exception as e:
    # Fallback if run outside the specific Colab environment
    print("Built-in Colab module unavailable. We will transition to the standard Gemini API in the next step.")
    print(f"Error details: {e}")

While the built-in Colab helper is convenient for quick validation, production-grade systems require formal authentication.

> To build a robust integration, **you need your own API key to manage quotas**, select specific model architectures, and configure operational parameters.

### 🔑 Get a (free) Gemini API key

1. Open **[Google AI Studio → API keys](https://aistudio.google.com/apikey)** and sign in with any Google account.
2. Click **Create API key** → copy it. *(Free tier — no credit card needed.)*
3. Back in Colab, click the **🔑 Secrets** icon in the left sidebar.
4. **Add new secret** → Name: `GOOGLE_API_KEY` → Value: *paste your key*.
5. Flip **Notebook access** ON for it to allow the code to read this variable.

### ⚠️ Critical Security Practice: Credential Management

You must never hardcode API keys directly into your source code—not in variables, not in comments, and not even "just for quick testing."

(Your future security team thanks you.)

### The Code: Initializing and Calling the Microservice

In [ ]:
# ---------------------------------------------------------
# 1. Install the Current SDK
# ---------------------------------------------------------
# Note: Ensure you are using 'google-genai'.
# The older 'google-generativeai' package is deprecated.
%pip install -q -U google-genai

from google import genai
from google.colab import userdata

# ---------------------------------------------------------
# 2. Authenticate and Initialize the Client
# ---------------------------------------------------------
# Securely fetching the API key from Colab's Secrets manager
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
print("Client successfully authenticated and ready. \n")

In [ ]:
# ---------------------------------------------------------
# 3. Execute the API Call
# ---------------------------------------------------------
# We are using the 'flash' tier, optimized for high speed and low latency.
MODEL = "gemini-3.5-flash"

print("Sending request to the model...\n")
response = client.models.generate_content(
    model=MODEL,
    contents="Explain Agentic AI to a software architect, in 2 sentences."
)

print("Response Received:\n")
print(response.text)

# The Final Architectural Takeaway

We have successfully opened the *black box*, understood the foundational mechanics inside, and closed it back up. The model is no longer an abstract concept —it is simply a measurable system of code and energy. It is now just another well-behaved, **scalable component in your system architecture**, ready to power the applications you build next.

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 July 11, Saturday*
